# Data Cleaning

This notebook applies deterministic data-quality corrections identified during
exploratory data analysis.

The cleaning stage operates on the complete raw dataset before the final
train/test split.

Primary goals:

- normalize inconsistent text representation;
- investigate and resolve duplicate project records;
- preserve and document genuine missing values;
- validate reward-level consistency;
- validate numerical values and dates;
- preserve legitimate extreme observations;
- produce a reproducible cleaned dataset for later preprocessing and feature
  engineering.

This notebook does **not** perform model-dependent preprocessing such as
scaling, encoding, statistical imputation, or target-based feature selection.

In [1]:
import sys
from pathlib import Path
import html
import re
import numpy as np
import pandas as pd
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.get_google_drive_data import load_csv_from_drive
from utils.google_drive_client import get_drive_client

## Load Raw Data

Cleaning begins from a fresh copy of the shared raw dataset rather than from the
dataframes modified during EDA.

The original dataframe is retained unchanged and a separate working copy is
created for cleaning.

In [2]:
service = get_drive_client()
df = load_csv_from_drive(service)

In [3]:
clean_df = df.copy(deep=True)

In [4]:
print("Raw shape:", df.shape)
print("Working copy shape:", clean_df.shape)

Raw shape: (45957, 17)
Working copy shape: (45957, 17)


## Baseline Data-Quality Audit

Before cleaning, record the original dataset shape, schema, missingness, and
duplicate structure. These values provide a baseline for comparison after
cleaning.

In [5]:
print(f"Rows: {len(clean_df):,}")
print(f"Columns: {len(clean_df.columns)}")
print(f"Unique project IDs: {clean_df['project id'].nunique():,}")

print(
    "Exact duplicate rows beyond first:",
    clean_df.duplicated().sum()
)

print(
    "Repeated project-ID rows beyond first:",
    clean_df["project id"].duplicated().sum()
)

Rows: 45,957
Columns: 17
Unique project IDs: 45,815
Exact duplicate rows beyond first: 89
Repeated project-ID rows beyond first: 142


In [6]:
baseline_schema = pd.DataFrame({
    "dtype": clean_df.dtypes.astype(str),
    "non-null": clean_df.notna().sum(),
    "missing": clean_df.isna().sum(),
    "missing %": (clean_df.isna().mean() * 100).round(2),
    "unique": clean_df.nunique(dropna=True),
})

baseline_schema

,dtype,non-null,missing,missing %,unique
project id,int64,45957,0,0.00,45815
name,object,45957,0,0.00,45754
url,object,45957,0,0.00,45814
category,object,45957,0,0.00,14
subcategory,object,45957,0,0.00,51
location,object,44635,1322,2.88,4849
status,object,45957,0,0.00,5
goal,float64,45957,0,0.00,1837
pledged,float64,45945,12,0.03,10950
funded percentage,float64,45957,0,0.00,21976


In [7]:
clean_df.describe()

,project id,goal,pledged,funded percentage,backers,levels,updates,comments,duration
count,4.595700e+04,4.595700e+04,4.594500e+04,45957.000000,45957.000000,45957.000000,45957.00000,45957.000000,45957.000000
mean,1.080800e+09,1.194271e+04,4.980750e+03,1.850129,69.973192,8.004939,4.08508,8.379529,39.995547
std,6.218057e+08,1.887583e+05,5.674162e+04,88.492706,688.628479,4.233907,6.43922,174.015737,17.414458
min,3.940900e+04,1.000000e-02,0.000000e+00,0.000000,0.000000,0.000000,0.00000,0.000000,1.000000
25%,5.438962e+08,1.800000e+03,1.960000e+02,0.044000,5.000000,5.000000,0.00000,0.000000,30.000000
50%,1.078345e+09,4.000000e+03,1.310000e+03,1.000000,23.000000,7.000000,2.00000,0.000000,32.000000
75%,1.621596e+09,9.862000e+03,4.165000e+03,1.115640,59.000000,10.000000,6.00000,3.000000,48.390000
max,2.147460e+09,2.147484e+07,1.026684e+07,15066.000000,87142.000000,80.000000,149.00000,19311.000000,91.960000


## Missing Values

Missing values are identified before deciding how they should be handled.

A missing value is not automatically an error and should not automatically be
imputed. Its meaning depends on the feature.

In [8]:
missing_summary = pd.DataFrame({
    "missing": clean_df.isna().sum(),
    "missing %": (clean_df.isna().mean() * 100).round(2),
})

missing_summary[missing_summary["missing"] > 0]

,missing,missing %
location,1322,2.88
pledged,12,0.03
reward levels,59,0.13


In [9]:
object_columns = clean_df.select_dtypes(include="object").columns

blank_counts = {}

for col in object_columns:
    blank_counts[col] = (
        clean_df[col]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

pd.Series(blank_counts, name="blank strings").sort_values(ascending=False)

name             0
url              0
category         0
subcategory      0
location         0
status           0
funded date      0
reward levels    0
Name: blank strings, dtype: int64

## Normalize Text Representation

EDA identified inconsistent HTML encoding in human-readable text fields.

Examples include:

- `Film &amp; Video` and `Film & Video`
- `Country &amp; Folk` and `Country & Folk`
- `&quot;Chased Resignation&quot;`

These differences represent formatting inconsistencies rather than meaningful
differences and should be normalized before duplicate resolution.

In [10]:
category_unique_before = clean_df["category"].nunique(dropna=True)
subcategory_unique_before = clean_df["subcategory"].nunique(dropna=True)

print("Categories before:", category_unique_before)
print("Subcategories before:", subcategory_unique_before)

Categories before: 14
Subcategories before: 51


In [11]:
sorted(clean_df["category"].dropna().unique())

['Art',
 'Comics',
 'Dance',
 'Design',
 'Fashion',
 'Film & Video',
 'Film &amp; Video',
 'Food',
 'Games',
 'Music',
 'Photography',
 'Publishing',
 'Technology',
 'Theater']

In [12]:
sorted(clean_df["subcategory"].dropna().unique())

['Animation',
 'Art',
 'Art Book',
 'Board & Card Games',
 'Board &amp; Card Games',
 "Children's Book",
 'Classical Music',
 'Comics',
 'Conceptual Art',
 'Country & Folk',
 'Country &amp; Folk',
 'Crafts',
 'Dance',
 'Design',
 'Digital Art',
 'Documentary',
 'Electronic Music',
 'Fashion',
 'Fiction',
 'Film &amp; Video',
 'Food',
 'Games',
 'Graphic Design',
 'Hip-Hop',
 'Illustration',
 'Indie Rock',
 'Jazz',
 'Journalism',
 'Mixed Media',
 'Music',
 'Narrative Film',
 'Nonfiction',
 'Open Hardware',
 'Open Software',
 'Painting',
 'Performance Art',
 'Periodical',
 'Photography',
 'Poetry',
 'Pop',
 'Product Design',
 'Public Art',
 'Publishing',
 'Rock',
 'Sculpture',
 'Short Film',
 'Technology',
 'Theater',
 'Video Games',
 'Webseries',
 'World Music']

In [13]:
TEXT_COLUMNS = [
    "name",
    "category",
    "subcategory",
    "location",
    "status",
]

for col in TEXT_COLUMNS:
    clean_df[col] = clean_df[col].apply(
        lambda value: html.unescape(value)
        if isinstance(value, str)
        else value
    )

    clean_df[col] = (
        clean_df[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    clean_df[col] = clean_df[col].replace("", pd.NA)

In [14]:
print(
    "Categories:",
    category_unique_before,
    "->",
    clean_df["category"].nunique(dropna=True)
)

print(
    "Subcategories:",
    subcategory_unique_before,
    "->",
    clean_df["subcategory"].nunique(dropna=True)
)

Categories: 14 -> 13
Subcategories: 51 -> 49


In [15]:
sorted(clean_df["category"].dropna().unique())

['Art',
 'Comics',
 'Dance',
 'Design',
 'Fashion',
 'Film & Video',
 'Food',
 'Games',
 'Music',
 'Photography',
 'Publishing',
 'Technology',
 'Theater']

In [16]:
sorted(clean_df["subcategory"].dropna().unique())

['Animation',
 'Art',
 'Art Book',
 'Board & Card Games',
 "Children's Book",
 'Classical Music',
 'Comics',
 'Conceptual Art',
 'Country & Folk',
 'Crafts',
 'Dance',
 'Design',
 'Digital Art',
 'Documentary',
 'Electronic Music',
 'Fashion',
 'Fiction',
 'Film & Video',
 'Food',
 'Games',
 'Graphic Design',
 'Hip-Hop',
 'Illustration',
 'Indie Rock',
 'Jazz',
 'Journalism',
 'Mixed Media',
 'Music',
 'Narrative Film',
 'Nonfiction',
 'Open Hardware',
 'Open Software',
 'Painting',
 'Performance Art',
 'Periodical',
 'Photography',
 'Poetry',
 'Pop',
 'Product Design',
 'Public Art',
 'Publishing',
 'Rock',
 'Sculpture',
 'Short Film',
 'Technology',
 'Theater',
 'Video Games',
 'Webseries',
 'World Music']

## Character-Decoding Audit

The raw source contains malformed byte sequences that are replaced during
loading when they cannot be decoded as valid UTF-8.

Rows containing the Unicode replacement character are identified so that
unrecoverable text corruption is not silently ignored.

In [17]:
replacement_character_counts = {}

for col in clean_df.select_dtypes(include=["object", "string"]).columns:
    count = (
        clean_df[col]
        .astype("string")
        .str.contains("\ufffd", regex=False, na=False)
        .sum()
    )

    if count > 0:
        replacement_character_counts[col] = count

pd.Series(
    replacement_character_counts,
    name="rows containing replacement character"
)

name        677
location    148
Name: rows containing replacement character, dtype: int64

## Duplicate Project Records

EDA found repeated project IDs. Some repeated records differed only in
`category` or `subcategory`, which may have resulted from inconsistent HTML
encoding.

Duplicates are therefore re-evaluated after text normalization before any rows
are removed.

In [18]:
duplicate_projects = (
    clean_df[
        clean_df["project id"].duplicated(keep=False)
    ]
    .sort_values("project id")
)

print(
    "Repeated project-ID rows beyond first:",
    clean_df["project id"].duplicated().sum()
)

print(
    "Distinct project IDs involved in duplicates:",
    duplicate_projects["project id"].nunique()
)

print(
    "Exact duplicate rows beyond first:",
    clean_df.duplicated().sum()
)

Repeated project-ID rows beyond first: 142
Distinct project IDs involved in duplicates: 142
Exact duplicate rows beyond first: 142


In [19]:
duplicate_conflicts = (
    duplicate_projects
    .groupby("project id")
    .nunique(dropna=False)
)

duplicate_conflicts.max().sort_values(ascending=False)

name                 1
url                  1
category             1
subcategory          1
location             1
status               1
goal                 1
pledged              1
funded percentage    1
backers              1
funded date          1
levels               1
reward levels        1
updates              1
comments             1
duration             1
dtype: int64

In [20]:
conflicting_project_ids = duplicate_conflicts[
    (duplicate_conflicts > 1).any(axis=1)
]

print(
    "Project IDs with remaining conflicting values:",
    len(conflicting_project_ids)
)

conflicting_project_ids.head(20)

Project IDs with remaining conflicting values: 0


,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
project id,,,,,,,,,,,,,,,,


In [21]:
if len(conflicting_project_ids) != 0:
    raise ValueError(
        "Conflicting duplicate project IDs remain. "
        "Inspect them before deduplication."
    )

In [22]:
rows_before_dedup = len(clean_df)

clean_df = (
    clean_df
    .drop_duplicates(subset="project id", keep="first")
    .reset_index(drop=True)
    .copy()
)

rows_after_dedup = len(clean_df)

print("Rows before deduplication:", rows_before_dedup)
print("Rows after deduplication:", rows_after_dedup)
print("Rows removed:", rows_before_dedup - rows_after_dedup)

Rows before deduplication: 45957
Rows after deduplication: 45815
Rows removed: 142


In [23]:
print(
    "Duplicate project IDs remaining:",
    clean_df["project id"].duplicated().sum()
)

print(
    "Exact duplicate rows remaining:",
    clean_df.duplicated().sum()
)

Duplicate project IDs remaining: 0
Exact duplicate rows remaining: 0


### Duplicate Resolution Result

The raw dataset contained 142 repeated project records beyond their first
occurrence.

Before text normalization, only 89 of these records were exact duplicates.
After HTML decoding and deterministic text normalization, all 142 repeated
records became exact duplicates and no project IDs contained conflicting
values.

This confirms that the apparent differences among repeated projects resulted
from inconsistent text representation rather than different project data.

One record was therefore retained for each `project id`, removing 142 duplicate
rows and leaving 45,815 unique projects.

## URL Uniqueness Audit

Project ID is the primary record identifier, but project URLs are also checked
for repeated values.

A repeated URL is investigated rather than automatically removed because a
shared URL does not by itself prove that two records are duplicates.

In [24]:
duplicate_urls = (
    clean_df[
        clean_df["url"].duplicated(keep=False)
    ]
    .sort_values("url")
)

print(
    "Rows belonging to repeated URLs:",
    len(duplicate_urls)
)

print(
    "Distinct repeated URLs:",
    duplicate_urls["url"].nunique()
)

duplicate_urls

Rows belonging to repeated URLs: 2
Distinct repeated URLs: 1


,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
13655,648882926,imAnerd Clothing: It's a new NerdStyle Brand,http://www.kickstarter.com/projects/1017911160...,Fashion,Fashion,"Tulsa, OK",live,1500.0,133.0,0.088667,5,"Sun, 17 Jun 2012 04:59:00 -0000",9,"$1,$5,$20,$30,$55,$100,$175,$500,$1,000",2,0,25.91
41011,1923124526,A More Perfect Union,http://www.kickstarter.com/projects/1017911160...,Film & Video,Webseries,"Henderson, NV",live,20000.0,1565.0,0.078250,10,"Sun, 01 Jul 2012 07:00:00 -0000",14,"$1,$5,$10,$15,$25,$35,$50,$75,$100,$150,$250,$...",9,0,36.41


In [25]:
duplicate_url_comparison = (
    duplicate_urls
    .groupby("url")
    .nunique(dropna=False)
)

duplicate_url_comparison.T

url,http://www.kickstarter.com/projects/1017911160/a-more-perfect-union
project id,2
name,2
category,2
subcategory,2
location,2
status,1
goal,2
pledged,2
funded percentage,2
backers,2


In [26]:
with pd.option_context("display.max_columns", None):
    display(duplicate_urls)

,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
13655,648882926,imAnerd Clothing: It's a new NerdStyle Brand,http://www.kickstarter.com/projects/1017911160...,Fashion,Fashion,"Tulsa, OK",live,1500.0,133.0,0.088667,5,"Sun, 17 Jun 2012 04:59:00 -0000",9,"$1,$5,$20,$30,$55,$100,$175,$500,$1,000",2,0,25.91
41011,1923124526,A More Perfect Union,http://www.kickstarter.com/projects/1017911160...,Film & Video,Webseries,"Henderson, NV",live,20000.0,1565.0,0.078250,10,"Sun, 01 Jul 2012 07:00:00 -0000",14,"$1,$5,$10,$15,$25,$35,$50,$75,$100,$150,$250,$...",9,0,36.41


## Re-evaluate Missing Values

Missing values are recalculated after duplicate resolution.

Missingness is preserved unless a deterministic replacement can be justified.
Statistical imputation is deferred to preprocessing so that learned values can
be fitted using training data only.

In [27]:
missing_after_dedup = pd.DataFrame({
    "missing": clean_df.isna().sum(),
    "missing %": (
        clean_df.isna().mean() * 100
    ).round(2),
})

missing_after_dedup[
    missing_after_dedup["missing"] > 0
]

,missing,missing %
location,1318,2.88
pledged,12,0.03
reward levels,59,0.13


In [28]:
missing_pledged = (
    clean_df[
        clean_df["pledged"].isna()
    ]
    .copy()
)

missing_pledged[
    [
        "project id",
        "name",
        "status",
        "goal",
        "pledged",
        "funded percentage",
        "backers",
    ]
]

,project id,name,status,goal,pledged,funded percentage,backers
1187,55442017,Xenonauts,live,50000.0,NaN,2.219487,3276
4502,213683990,Twokinds Book Printing Drive,live,25000.0,NaN,6.303783,1855
13381,636029584,HICKIES - TURN YOUR KICKS INTO SLIP-ONS,live,25000.0,NaN,4.907843,2707
13802,656206605,Genie - Motion control time lapse device,live,150000.0,NaN,2.999618,671
25239,1184761342,B9Creator - A High Resolution 3D Printer,live,50000.0,NaN,4.761380,233
29412,1382865280,"Phil Tippett's ""MAD GOD""",live,40000.0,NaN,2.532484,1950
31164,1467412732,gTar: The First Guitar That Anybody Can Play,live,100000.0,NaN,2.846420,753
34274,1612550316,Space Command,live,75000.0,NaN,1.796187,1610
35032,1648656751,BronyCon: The Documentary,live,60000.0,NaN,3.148185,1540
40759,1911111579,Two Guys SpaceVenture - by the creators of Spa...,live,500000.0,NaN,0.583738,5473


In [29]:
missing_pledged["implied pledged"] = (
    missing_pledged["goal"]
    * missing_pledged["funded percentage"]
)

missing_pledged[
    [
        "project id",
        "status",
        "goal",
        "pledged",
        "funded percentage",
        "implied pledged",
        "backers",
    ]
]

,project id,status,goal,pledged,funded percentage,implied pledged,backers
1187,55442017,live,50000.0,NaN,2.219487,110974.340000,3276
4502,213683990,live,25000.0,NaN,6.303783,157594.580000,1855
13381,636029584,live,25000.0,NaN,4.907843,122696.070000,2707
13802,656206605,live,150000.0,NaN,2.999618,449942.770050,671
25239,1184761342,live,50000.0,NaN,4.761380,238069.010000,233
29412,1382865280,live,40000.0,NaN,2.532484,101299.360000,1950
31164,1467412732,live,100000.0,NaN,2.846420,284642.010000,753
34274,1612550316,live,75000.0,NaN,1.796187,134714.040000,1610
35032,1648656751,live,60000.0,NaN,3.148185,188891.130000,1540
40759,1911111579,live,500000.0,NaN,0.583738,291868.760000,5473


## Validate Reward-Level Data

The reward-level parser developed during EDA is validated on the complete,
deduplicated dataset.

The purpose of this step is to verify data integrity rather than create derived
features such as `tier_step`.

In [30]:
def parse_reward_levels(value):
    if pd.isna(value):
        return []

    amounts = re.findall(
        r"\$(\d+(?:,\d{3})*(?:\.\d+)?)",
        str(value)
    )

    return [
        float(amount.replace(",", ""))
        for amount in amounts
    ]

In [31]:
parsed_rewards = (
    clean_df["reward levels"]
    .apply(parse_reward_levels)
)

parsed_level_counts = (
    parsed_rewards
    .apply(len)
)

level_matches = (
    clean_df["levels"]
    == parsed_level_counts
)

level_matches.value_counts()

True    45815
Name: count, dtype: int64

In [32]:
print(
    "Reward-level count mismatches:",
    (~level_matches).sum()
)

clean_df.loc[
    ~level_matches,
    [
        "project id",
        "levels",
        "reward levels",
    ]
].head(20)

Reward-level count mismatches: 0


,project id,levels,reward levels


In [33]:
missing_reward_levels = (
    clean_df[
        clean_df["reward levels"].isna()
    ]
)

print(
    "Missing reward-level strings:",
    len(missing_reward_levels)
)

print(
    "Missing strings with levels == 0:",
    missing_reward_levels["levels"].eq(0).sum()
)

Missing reward-level strings: 59
Missing strings with levels == 0: 59


### Data-Integrity Findings

The URL audit identified one URL shared by two different project records.
The projects differ across their IDs, names, categories, goals, funding
information, dates, and reward structures. Therefore, URL is not treated as a
unique project identifier and neither project is removed.

After duplicate resolution, missing values remain in `location`, `pledged`,
and `reward levels`.

All 12 missing `pledged` values belong to campaigns whose status is `live`.
These records contain nonzero backer counts and funded percentages, indicating
that the missing values should not be interpreted as zero pledged amounts.
They are therefore preserved as missing source values.

Reward-level validation succeeded for all 45,815 projects. The parsed number
of reward amounts exactly matches the provided `levels` column for every row.
All 59 missing `reward levels` values correspond to `levels == 0`, indicating
that these missing strings consistently represent campaigns with no reward
tiers rather than parsing failures.

## Numerical Validity Checks

Extreme numerical values are not automatically treated as errors.

The cleaning stage checks explicit domain constraints and non-finite values
while preserving unusual but potentially legitimate Kickstarter campaigns.

In [34]:
numerical_validity = pd.Series({
    "goal <= 0":
        clean_df["goal"].le(0).sum(),

    "pledged < 0":
        clean_df["pledged"].lt(0).sum(),

    "funded percentage < 0":
        clean_df["funded percentage"].lt(0).sum(),

    "backers < 0":
        clean_df["backers"].lt(0).sum(),

    "levels < 0":
        clean_df["levels"].lt(0).sum(),

    "updates < 0":
        clean_df["updates"].lt(0).sum(),

    "comments < 0":
        clean_df["comments"].lt(0).sum(),

    "duration <= 0":
        clean_df["duration"].le(0).sum(),
}, name="invalid rows")

numerical_validity

goal <= 0                0
pledged < 0              0
funded percentage < 0    0
backers < 0              0
levels < 0               0
updates < 0              0
comments < 0             0
duration <= 0            0
Name: invalid rows, dtype: int64

In [35]:
numeric_columns = clean_df.select_dtypes(
    include=np.number
).columns

infinite_counts = pd.Series({
    col: np.isinf(clean_df[col]).sum()
    for col in numeric_columns
})

infinite_counts[infinite_counts > 0]

Series([], dtype: int64)

In [36]:
clean_df[
    [
        "goal",
        "pledged",
        "funded percentage",
        "backers",
        "levels",
        "updates",
        "comments",
        "duration",
    ]
].describe()

,goal,pledged,funded percentage,backers,levels,updates,comments,duration
count,4.581500e+04,4.580300e+04,45815.000000,45815.00000,45815.000000,45815.000000,45815.000000,45815.000000
mean,1.195542e+04,4.985739e+03,1.847629,70.04217,8.003972,4.083466,8.396093,39.995114
std,1.890487e+05,5.682869e+04,88.620172,689.68463,4.234317,6.438860,174.284421,17.420618
min,1.000000e-02,0.000000e+00,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000
25%,1.800000e+03,1.950000e+02,0.044000,5.00000,5.000000,0.000000,0.000000,30.000000
50%,4.000000e+03,1.310000e+03,1.000000,23.00000,7.000000,2.000000,0.000000,32.000000
75%,9.884000e+03,4.160000e+03,1.115677,59.00000,10.000000,6.000000,3.000000,48.395000
max,2.147484e+07,1.026684e+07,15066.000000,87142.00000,80.000000,149.000000,19311.000000,91.960000


### Outlier Treatment Decision

The numerical distributions contain extreme observations, but the validity
checks do not show impossible values or infinities.

Earlier machine-learning work used winsorization to reduce the influence of
extreme observations. Winsorization is not applied automatically here because
large Kickstarter goals, pledged amounts, backer counts, and engagement values
may represent legitimate campaigns rather than data errors.

If later modeling shows that heavy-tailed variables harm performance,
transformations such as `log1p`, clipping, or winsorization can be evaluated
inside the modeling pipeline using training data only.

## Validate Funded Percentage

For records where `pledged` is available, the supplied `funded percentage`
value is compared with the value implied by `pledged / goal`.

The two values are not assumed to be numerically identical. Differences may
result from rounding, source timing, or other conventions in the original
dataset.

This section therefore examines the magnitude of the differences rather than
automatically classifying every non-identical value as erroneous.

No source value is overwritten during this integrity check.

In [37]:
funded_check_mask = (
    clean_df["pledged"].notna()
    & clean_df["goal"].gt(0)
)

expected_funded_percentage = (
    clean_df.loc[funded_check_mask, "pledged"]
    / clean_df.loc[funded_check_mask, "goal"]
)

provided_funded_percentage = (
    clean_df.loc[
        funded_check_mask,
        "funded percentage",
    ]
)

funded_consistency = pd.DataFrame({
    "provided": provided_funded_percentage,
    "expected": expected_funded_percentage,
})

funded_consistency["difference"] = (
    funded_consistency["provided"]
    - funded_consistency["expected"]
)

funded_consistency["absolute difference"] = (
    funded_consistency["difference"].abs()
)

print("Rows checked:", len(funded_consistency))

funded_consistency[
    "absolute difference"
].describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

Rows checked: 45803


count    45803.000000
mean         0.000039
std          0.003219
min          0.000000
50%          0.000000
90%          0.000015
95%          0.000068
99%          0.000333
99.9%        0.001524
max          0.500000
Name: absolute difference, dtype: float64

In [38]:
for tolerance in [
    1e-6,
    1e-5,
    1e-4,
    1e-3,
    1e-2,
]:
    within_tolerance = (
        funded_consistency[
            "absolute difference"
        ]
        <= tolerance
    ).mean()

    print(
        f"Within {tolerance:g}: "
        f"{within_tolerance:.2%}"
    )

Within 1e-06: 86.27%
Within 1e-05: 89.07%
Within 0.0001: 96.40%
Within 0.001: 99.81%
Within 0.01: 99.98%


In [39]:
largest_funded_differences = (
    funded_consistency
    .nlargest(
        20,
        "absolute difference"
    )
)

clean_df.loc[
    largest_funded_differences.index,
    [
        "project id",
        "status",
        "goal",
        "pledged",
        "funded percentage",
    ]
].assign(
    expected_funded_percentage=(
        largest_funded_differences["expected"]
    ),
    absolute_difference=(
        largest_funded_differences[
            "absolute difference"
        ]
    ),
)

,project id,status,goal,pledged,funded percentage,expected_funded_percentage,absolute_difference
29868,1404938334,successful,1.0,2867.0,2866.500000,2867.000000,0.500000
14561,690081674,successful,1.0,1722.0,1722.470000,1722.000000,0.470000
336,15438342,successful,10.0,148.0,14.750000,14.800000,0.050000
23445,1100872151,successful,25.0,26244.0,1049.743200,1049.760000,0.016800
3617,171764814,successful,36.0,115.0,3.182500,3.194444,0.011944
29902,1406372094,successful,42.0,414.0,9.867619,9.857143,0.010476
44681,2096069537,successful,50.0,103.0,2.050000,2.060000,0.010000
45743,2144102391,successful,15.0,4202.0,280.142667,280.133333,0.009333
44946,2108610803,successful,75.0,113.0,1.500000,1.506667,0.006667
7314,346562116,successful,50.0,52.0,1.046600,1.040000,0.006600


### Funded-Percentage Treatment Decision

The consistency audit is diagnostic only.

Because the supplied `funded percentage` field and the value calculated from
`pledged / goal` may reflect different rounding or snapshot conventions, small
differences are not treated as data errors. The largest discrepancies should be
reviewed before any correction is considered.

The original source values are preserved in the cleaned dataset.

## Categorical Consistency

Categorical values are checked after HTML decoding to verify that normalization
did not create unexpected category structures.

In [40]:
clean_df["status"].value_counts(dropna=False)

status
successful    22902
failed        18939
live           3912
canceled         58
suspended         4
Name: count, dtype: Int64

In [41]:
sorted(clean_df["category"].dropna().unique())

['Art',
 'Comics',
 'Dance',
 'Design',
 'Fashion',
 'Film & Video',
 'Food',
 'Games',
 'Music',
 'Photography',
 'Publishing',
 'Technology',
 'Theater']

In [42]:
subcategory_parent_counts = (
    clean_df
    .groupby("subcategory")["category"]
    .nunique()
    .sort_values(ascending=False)
)

subcategory_parent_counts[
    subcategory_parent_counts > 1
]

Series([], Name: category, dtype: int64)

In [43]:
html_entity_pattern = (
    r"&(?:#\d+|#x[0-9A-Fa-f]+|"
    r"[A-Za-z][A-Za-z0-9]+);"
)

remaining_html_entities = {}

for col in TEXT_COLUMNS:
    remaining_html_entities[col] = (
        clean_df[col]
        .astype("string")
        .str.contains(
            html_entity_pattern,
            regex=True,
            na=False,
        )
        .sum()
    )

pd.Series(
    remaining_html_entities,
    name="remaining HTML entities"
)

name           0
category       0
subcategory    0
location       0
status         0
Name: remaining HTML entities, dtype: int64

## Validate Funded Dates

The raw `funded date` field is stored as text.

Dates are parsed to confirm that they are valid and are then standardized to a
consistent UTC representation.

In [44]:
parsed_funded_date = pd.to_datetime(
    clean_df["funded date"],
    errors="coerce",
    utc=True,
)

print(
    "Date parsing failures:",
    parsed_funded_date.isna().sum()
)

print(
    "Earliest funded date:",
    parsed_funded_date.min()
)

print(
    "Latest funded date:",
    parsed_funded_date.max()
)

Date parsing failures: 0
Earliest funded date: 2009-05-03 06:59:59+00:00
Latest funded date: 2012-08-01 19:36:12+00:00


In [45]:
clean_df.loc[
    parsed_funded_date.isna(),
    ["project id", "funded date"]
].head(20)

,project id,funded date


In [46]:
if parsed_funded_date.isna().sum() != 0:
    raise ValueError(
        "Some funded dates could not be parsed."
    )

clean_df["funded date"] = (
    parsed_funded_date
    .dt.strftime("%Y-%m-%dT%H:%M:%SZ")
)

## Normalize Location Formatting

Location values contain genuine missingness and inconsistent capitalization of
some trailing two-letter abbreviations.

Only deterministic formatting is standardized. Missing geographic information
is not inferred, and a derived `state` feature is deferred to feature
engineering.

In [47]:
location_suffixes = (
    clean_df["location"]
    .str.extract(
        r",\s*([A-Za-z]{2})\s*$"
    )[0]
)

mixed_case_suffixes = (
    location_suffixes[
        location_suffixes.notna()
        & location_suffixes.ne(
            location_suffixes.str.upper()
        )
    ]
)

mixed_case_suffixes.value_counts()

0
Mt    36
Name: count, dtype: Int64

In [48]:
def normalize_location(value):
    if pd.isna(value):
        return value

    return re.sub(
        r",\s*([A-Za-z]{2})\s*$",
        lambda match: (
            ", " + match.group(1).upper()
        ),
        value,
    )


clean_df["location"] = (
    clean_df["location"]
    .apply(normalize_location)
)

In [49]:
location_suffixes_after = (
    clean_df["location"]
    .str.extract(
        r",\s*([A-Za-z]{2})\s*$"
    )[0]
)

mixed_case_suffixes_after = (
    location_suffixes_after[
        location_suffixes_after.notna()
        & location_suffixes_after.ne(
            location_suffixes_after.str.upper()
        )
    ]
)

print(
    "Mixed-case two-letter suffixes remaining:",
    len(mixed_case_suffixes_after)
)

Mixed-case two-letter suffixes remaining: 0


## Character-Decoding Limitation

The source CSV contains malformed byte sequences that cannot be decoded
perfectly using the project's UTF-8 loading strategy.

Some project names and locations therefore contain the Unicode replacement
character (`�`).

These values are preserved rather than guessed or manually substituted because
the exact original character cannot be recovered reliably from the decoded
dataframe. This is treated as a documented source-data limitation.

In [50]:
replacement_character_counts_final = {}

for col in clean_df.select_dtypes(
    include=["object", "string"]
).columns:
    count = (
        clean_df[col]
        .astype("string")
        .str.contains(
            "\ufffd",
            regex=False,
            na=False,
        )
        .sum()
    )

    if count > 0:
        replacement_character_counts_final[col] = count

pd.Series(
    replacement_character_counts_final,
    name="rows containing replacement character"
)

name        676
location    148
Name: rows containing replacement character, dtype: int64

## Final Missingness Audit

Remaining missing values are reviewed after deterministic cleaning.

Missing values without a justified deterministic replacement remain missing
for later preprocessing decisions.

In [51]:
final_missing = pd.DataFrame({
    "missing": clean_df.isna().sum(),
    "missing %": (
        clean_df.isna().mean() * 100
    ).round(2),
})

final_missing[
    final_missing["missing"] > 0
]

,missing,missing %
location,1318,2.88
pledged,12,0.03
reward levels,59,0.13


## Final Dataset Validation

In [52]:
print("Final shape:", clean_df.shape)

print(
    "Unique project IDs:",
    clean_df["project id"].nunique()
)

print(
    "Duplicate project IDs:",
    clean_df["project id"].duplicated().sum()
)

print(
    "Exact duplicate rows:",
    clean_df.duplicated().sum()
)

print(
    "Categories:",
    clean_df["category"].nunique()
)

print(
    "Subcategories:",
    clean_df["subcategory"].nunique()
)

Final shape: (45815, 17)
Unique project IDs: 45815
Duplicate project IDs: 0
Exact duplicate rows: 0
Categories: 13
Subcategories: 49


In [53]:
assert clean_df["project id"].duplicated().sum() == 0
assert clean_df.duplicated().sum() == 0
assert clean_df["project id"].notna().all()
assert clean_df["category"].notna().all()
assert clean_df["subcategory"].notna().all()
assert clean_df["status"].notna().all()
assert df.shape == (45957, 17)

In [54]:
cleaning_summary = pd.DataFrame({
    "metric": [
        "rows",
        "unique project IDs",
        "exact duplicate rows",
        "repeated project-ID rows",
        "categories",
        "subcategories",
        "missing location",
        "missing pledged",
        "missing reward levels",
    ],

    "before": [
        len(df),
        df["project id"].nunique(),
        df.duplicated().sum(),
        df["project id"].duplicated().sum(),
        df["category"].nunique(),
        df["subcategory"].nunique(),
        df["location"].isna().sum(),
        df["pledged"].isna().sum(),
        df["reward levels"].isna().sum(),
    ],

    "after": [
        len(clean_df),
        clean_df["project id"].nunique(),
        clean_df.duplicated().sum(),
        clean_df["project id"].duplicated().sum(),
        clean_df["category"].nunique(),
        clean_df["subcategory"].nunique(),
        clean_df["location"].isna().sum(),
        clean_df["pledged"].isna().sum(),
        clean_df["reward levels"].isna().sum(),
    ],
})

cleaning_summary

,metric,before,after
0,rows,45957,45815
1,unique project IDs,45815,45815
2,exact duplicate rows,89,0
3,repeated project-ID rows,142,0
4,categories,14,13
5,subcategories,51,49
6,missing location,1322,1318
7,missing pledged,12,12
8,missing reward levels,59,59


## Save Cleaned Dataset

In [55]:
processed_dir = (
    PROJECT_ROOT
    / "data"
    / "Processed-Data"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

cleaned_path = (
    processed_dir
    / "DSI_kickstarterscrape_dataset_cleaned.csv"
)

clean_df.to_csv(
    cleaned_path,
    index=False,
)

print(
    "Saved cleaned dataset to:",
    cleaned_path
)

Saved cleaned dataset to: /Users/annasehgal/State-Street-1C-kickstarter-crowdfunding-recommendation-engine/data/Processed-Data/DSI_kickstarterscrape_dataset_cleaned.csv


In [56]:
saved_df = pd.read_csv(cleaned_path)

assert saved_df.shape == clean_df.shape
assert saved_df["project id"].duplicated().sum() == 0

print("Saved shape:", saved_df.shape)
print("Saved dataset validation passed.")

Saved shape: (45815, 17)
Saved dataset validation passed.


# Cleaning Completion and Next Steps

This notebook completes deterministic cleaning of the raw Kickstarter dataset.

The cleaned dataset retains 45,815 unique project records while preserving
source information that may require model-dependent treatment later.

## Cleaning completed

The cleaning workflow:

- loaded and preserved the complete raw dataset;
- recorded baseline shape, schema, missingness, and duplicate structure;
- distinguished true missing values from blank strings;
- decoded inconsistent HTML entities;
- normalized deterministic whitespace and text representation;
- identified malformed source characters without guessing replacements;
- investigated repeated project IDs before deleting records;
- confirmed that repeated projects became exact duplicates after text
  normalization;
- retained one record per unique project ID;
- investigated a repeated URL and confirmed that URL is not a reliable unique
  project identifier;
- investigated remaining missing values rather than automatically imputing them;
- validated reward-tier parsing against the existing `levels` field;
- checked numerical fields for impossible and non-finite values;
- preserved extreme but potentially valid campaign observations;
- evaluated the consistency of funded percentage with pledged amount and goal;
- verified categorical structure after text normalization;
- parsed and standardized funded dates;
- normalized deterministic location formatting;
- rechecked final missingness, schema, row counts, and uniqueness;
- saved and reloaded the cleaned dataset to verify the generated output.

## Missing-value treatment

Missing values are not automatically replaced during deterministic cleaning.

`location` remains missing when geographic information is unavailable.

The missing `pledged` values are preserved because they occur in live campaigns
that contain nonzero backer counts and funded percentages, so treating them as
zero would be incorrect.

Missing `reward levels` values are preserved because validation showed that
they consistently correspond to `levels == 0`.

Statistical imputation, if needed for modeling, should be learned only from the
training data.

## Outliers and extreme values

Earlier machine-learning work used techniques such as winsorization to reduce
the influence of extreme numerical observations.

For this dataset, extreme Kickstarter campaign values are not automatically
treated as errors. Campaign goals, pledged amounts, backers, and engagement can
legitimately vary by several orders of magnitude.

Therefore, the cleaning stage validates explicit domain constraints rather than
automatically clipping or winsorizing the data.

Transformations such as logarithmic scaling or winsorization may be evaluated
later if they improve a particular model.

## Deferred preprocessing and feature engineering

Several techniques belong in later stages rather than deterministic cleaning:

- **Statistical imputation:** fit using training data only.
- **Categorical missing-value representation:** an `Unknown` category may be
  introduced later if required by the modeling pipeline.
- **Missingness indicators:** features such as `location_missing` may be useful
  if missingness itself contains predictive information.
- **One-hot encoding:** categorical encoders should be fitted using training
  data and should handle unseen test categories safely.
- **Numerical scaling:** scalers must be fitted on training data and then
  applied unchanged to validation and test data.
- **Outlier transformations:** winsorization, clipping, or log transformations
  should be evaluated for modeling value rather than applied automatically.
- **Class-imbalance handling:** should be evaluated after defining the final
  settled-campaign modeling population.
- **Feature selection:** identifiers and variables unavailable at prediction
  time should be excluded from the final model feature set.
- **Derived features:** `tier_step`, standardized state, missingness indicators,
  and transformed numerical features belong in feature engineering.

## Information-leakage boundary

The cleaned dataset preserves the original source columns, but not every source
column should necessarily become a model input.

Variables including:

- `pledged`
- `funded percentage`
- `backers`
- `updates`
- `comments`

may contain information generated after a campaign launches.

If the intended task is to predict campaign success using launch-time
information, these features should be excluded from the model unless the
prediction point is explicitly defined differently.

`project id` and `url` are identifiers and should also not be treated as
ordinary predictive features.

## Final output

The cleaned dataset is a validated source-level dataset rather than a final
model matrix.

Model-dependent preprocessing, feature engineering, train/test separation, and
model fitting should be performed in later pipeline stages without altering the
raw source data.